In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import xtrack as xt
import xpart as xp
import xobjects as xo
import xplt
import gc
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex'])
import pickle
from datetime import datetime
from models import *

from scipy import stats
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

torch.set_default_dtype(torch.float64)

In [ ]:
def init_weights_glorot(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def normalize(x, minimum=None, maximum=None):
    if minimum is None:
        minimum = x.min()
    if maximum is None:
        maximum = x.max()
    return (x - minimum) / (maximum - minimum)

## Data preparation

In [ ]:
inputs = []

for parameter in [0.5095, 0.61, 0.71]:
    with open(f'data/simple_line_10k_perturbed_{parameter}.pkl', 'rb') as f:
        data = pickle.load(f)
        tmp_inputs = torch.tensor(np.array([data[0][:, :-1], data[1][:, :-1], data[2][:, 1:], data[3][:, 1:]]))
        inputs.append(tmp_inputs)

inputs = torch.cat(inputs, dim=1).permute(1, 0, 2)


inputs_test = []

for parameter in [0.57, 0.68]:
    with open(f'data/simple_line_10k_perturbed_{parameter}.pkl', 'rb') as f:
        data = pickle.load(f)
        tmp_inputs = torch.tensor(np.array([data[0][:, :-1], data[1][:, :-1], data[2][:, 1:], data[3][:, 1:]]))
        inputs_test.append(tmp_inputs)

inputs_test = torch.cat(inputs_test, dim=1).permute(1, 0, 2)

In [ ]:
gridsize = 32

np.random.seed(144)
n_samples = 3 * 10**5 
idx_particle = np.random.choice(range(inputs.shape[0]), n_samples, replace=True)
idx_turn = np.random.choice(range(gridsize, inputs.shape[2]-gridsize), n_samples, replace=True)

In [ ]:
inputs_train = []
targets_train = []
for i, j in zip(idx_particle, idx_turn):
    inputs_train.append(inputs[i:i+1, :, j-gridsize:j])
    targets_train.append(inputs[i:i+1, :2, j:j+gridsize])

inputs_train = torch.cat(inputs_train, dim=0)
targets_train = torch.cat(targets_train, dim=0)

minimum_list = []
maximum_list = []
for i in range(inputs_train.shape[1]):
    minimum = inputs_train[:, i, :].min()
    maximum = inputs_train[:, i, :].max()

    minimum_list.append(minimum)
    maximum_list.append(maximum)

    inputs_train[:, i, :] = normalize(inputs_train[:, i, :], minimum, maximum)
    if i < targets_train.shape[1]:
        targets_train[:, i, :] = normalize(targets_train[:, i, :], minimum, maximum)

torch.save((inputs_train, targets_train, minimum_list, maximum_list), f'data/data_train_simple_line_fno_{gridsize}.pt')
del inputs_train
del targets_train
gc.collect()

inputs_train_ar = []
targets_train_ar = []
#idx_particle = np.random.choice(range(inputs.shape[0]), 200, replace=False)
for i in range(inputs.shape[0]):
    for j in range(gridsize, inputs.shape[2]-gridsize, gridsize):
        inputs_train_ar.append(inputs[i:i+1, :, j-gridsize:j])
        targets_train_ar.append(inputs[i:i+1, :2, j:j+gridsize])

inputs_train_ar = torch.cat(inputs_train_ar, dim=0)
targets_train_ar = torch.cat(targets_train_ar, dim=0)

for i in range(inputs_train_ar.shape[1]):
    minimum = minimum_list[i]
    maximum = maximum_list[i]

    inputs_train_ar[:, i, :] = normalize(inputs_train_ar[:, i, :], minimum, maximum)
    if i < targets_train_ar.shape[1]:
        targets_train_ar[:, i, :] = normalize(targets_train_ar[:, i, :], minimum, maximum)

torch.save((inputs_train_ar, targets_train_ar, minimum_list, maximum_list), f'data/data_train_eval_simple_line_fno_{gridsize}.pt')
del inputs_train_ar
del targets_train_ar
gc.collect()

inputs_test_ar = []
targets_test_ar = []
#idx_particle = np.random.choice(range(inputs.shape[0]), 200, replace=False)
for i in range(inputs_test.shape[0]):
    for j in range(gridsize, inputs_test.shape[2]-gridsize, gridsize):
        inputs_test_ar.append(inputs_test[i:i+1, :, j-gridsize:j])
        targets_test_ar.append(inputs_test[i:i+1, :2, j:j+gridsize])

inputs_test_ar = torch.cat(inputs_test_ar, dim=0)
targets_test_ar = torch.cat(targets_test_ar, dim=0)


for i in range(inputs_test_ar.shape[1]):
    minimum = minimum_list[i]
    maximum = maximum_list[i]

    inputs_test_ar[:, i, :] = normalize(inputs_test_ar[:, i, :], minimum, maximum)
    if i < targets_test_ar.shape[1]:
        targets_test_ar[:, i, :] = normalize(targets_test_ar[:, i, :], minimum, maximum)

torch.save((inputs_test_ar, targets_test_ar, minimum_list, maximum_list), f'data/data_test_simple_line_fno_{gridsize}.pt')